# Footwear pattern classification under occlusion and cross-medium transfer
**Dora Zhuang · MGSC 673**

This notebook consolidates the completed Handiprint study with the tile/synthetic-blood extension. It compares two specific sole patterns (Adidas Seeley and Nike Zoom Winflo 4), not general manufacturer identification or wearer identity. The original notebooks and results remain preserved.

**Recorded findings:** severe simulated Handiprint masking exposed an augmentation benefit in the earlier exploratory study. On related-pair blood images, RGB, grayscale and occlusion-trained heads all reach the observed test accuracy ceiling; collection, size and crop controls limit pattern-specific interpretation. All extension results are exploratory because earlier test pairs and labelled diagnostics were already viewed.

Default execution verifies existing inputs, reuses the formal **GPU-trained** heads, reproduces or reuses fixed-model predictions, recomputes pair-grouped summaries, and writes figures and a concise report; a separate packager builds the submission after the notebook is saved. It does not repeat image acquisition, manual review, split generation or CPU-model selection. Existing raw inputs, canonical annotations, preprocessing lock and old run folders are read-only.

Run instructions, the optional re-training check and packaging steps are in README.md. The original metadata snapshot remains in the preparation notebook; this consolidated version reads its hash-locked audit rather than duplicating that large literal.


## 1. Portable data, dependencies and run modes
Extract the entire project and open this notebook from its directory. The bundle is detected automatically and contains all 308 prepared RGB224 images, saved formal GPU heads, exact source manifests, original training statistics, legacy weights and review histories. Google Drive and original full-resolution JPG/TIFF files are not required for the default evaluation.

Default RUN_TRAINING=False reuses hash-verified GPU models and shows their actual training histories. RUN_TRAINING=True reproduces fits in a separate results/blood_reproduction_v1 directory using the bundled pixels; it never replaces the formal models. CPU reproduction is supported but can be slow because augmentation requires repeated backbone evaluation. GPU is recommended for fresh fits. This project does not silently replace the original augmentation protocol by fitting once on precomputed features.

Install requirements.txt in a suitable Python environment. The first run may download the published ImageNet backbone (about 45 MB); its SHA is verified. Both training and inference versions are recorded. The original absolute crop paths remain in the immutable source manifest for provenance. portable_manifest.csv supplies relative paths, and the notebook resolves bundled inputs by split and key.

Optional re-training is switched on with the environment variable `FOOTWEAR_RUN_TRAINING=1` (see README.md) instead of editing this notebook, so the saved default run and the reproduction check use the same file. The re-training check should be run on a copy of the project; it writes its log and receipts to `results/blood_reproduction_v1` and never replaces the formal GPU models.


In [ ]:
from pathlib import Path
import os, json, hashlib, math, io, zipfile, base64, zlib
import numpy as np
import pandas as pd
from PIL import Image, ImageOps
from IPython.display import display, Markdown

# Auto-detect a freshly extracted project; no personal Drive is required.
_explicit_bundle=os.environ.get('FOOTWEAR_ANALYSIS_BUNDLE')
_bundle_candidates=[Path(_explicit_bundle)] if _explicit_bundle else [Path.cwd(),Path.cwd()/'MGSC673_Portable_Project']
BUNDLE_DIR=next((p.resolve() for p in _bundle_candidates if (p/'input/gpu').is_dir() and (p/'input/locked').is_dir()),None)
ROOT = Path(os.environ.get('FOOTWEAR_ROOT','/content/drive/MyDrive/FOOTWEAR_PRIVATE_PROJECT'))
if BUNDLE_DIR is None and str(ROOT).startswith('/content/drive/') and not ROOT.exists():
    from google.colab import drive
    drive.mount('/content/drive')
# Also find the new project when opened directly from the author's Drive.
if BUNDLE_DIR is None and not _explicit_bundle:
    _drive_bundle=ROOT/'results/blood_external_gpu_v1/portable_submission_v2/MGSC673_Portable_Project'
    if (_drive_bundle/'input/gpu').is_dir() and (_drive_bundle/'input/locked').is_dir():
        BUNDLE_DIR=_drive_bundle.resolve()
BASE = (BUNDLE_DIR/'input/locked') if BUNDLE_DIR else ROOT/'results/blood_tile/20261025T183821Z_90054831'
CANONICAL_RUN = BASE
GPU = (BUNDLE_DIR/'input/gpu') if BUNDLE_DIR else ROOT/'results/blood_external_gpu_v1'
FINAL_OUTPUT = (BUNDLE_DIR/'reproduced_analysis') if BUNDLE_DIR else GPU/'final_submission_v1'
TRAIN_STATISTICS = (BUNDLE_DIR/'input/train_canvas_statistics.csv') if BUNDLE_DIR else GPU/'analysis_handoff/train_canvas_statistics_31295e506f.csv'
LEGACY_ARCHIVE = (BUNDLE_DIR/'legacy/results_to_share.zip') if BUNDLE_DIR else ROOT/'results_to_share.zip'
NOTEBOOK_PATH = (BUNDLE_DIR/'MGSC673_Project.ipynb') if BUNDLE_DIR else ROOT/'notebooks/MGSC673_Project.ipynb'
RUN_TRAINING = os.environ.get('FOOTWEAR_RUN_TRAINING','0')=='1'   # default: reuse formal GPU models
REBUILD_INPUTS = False
SHOW_TRAIN_PREVIEWS = True
DOWNLOAD_SUBMISSION = False
assert GPU.is_dir() and BASE.is_dir(), 'Supply the documented locked run and formal GPU artifacts.'
assert TRAIN_STATISTICS.is_file(), f'Missing training summaries: {TRAIN_STATISTICS}'
FINAL_OUTPUT.mkdir(parents=True,exist_ok=True)
print('Formal models:',GPU)
print('Analysis/report output:',FINAL_OUTPUT)


In [ ]:
# Public ImageNet ResNet18 backbone: needed by the fixed-model reproduction and by optional re-training.
# It is looked up in the torch cache, then in an optional bundled copy, then downloaded; its SHA256 must match the training record.
import shutil, hashlib
import torch, torchvision

def ensure_imagenet_backbone():
    expected = json.loads((GPU/'environment.json').read_text()).get('backbone_weights_sha256')
    target = Path(torch.hub.get_dir())/'checkpoints'/'resnet18-f37072fd.pth'
    target.parent.mkdir(parents=True, exist_ok=True)
    bundled = (BUNDLE_DIR/'input/backbone/resnet18-f37072fd.pth') if BUNDLE_DIR else None
    if not target.is_file() and bundled is not None and bundled.is_file():
        shutil.copy2(bundled, target)
    if not target.is_file():
        try:
            torchvision.models.resnet18(weights=torchvision.models.ResNet18_Weights.IMAGENET1K_V1)
        except Exception as error:
            raise RuntimeError(
                'The public ImageNet ResNet18 backbone (about 45 MB) is required but could not be downloaded '
                f'({type(error).__name__}). Connect to the internet and rerun this cell, or place the file '
                f'resnet18-f37072fd.pth in {target.parent}'
                + (f' (expected SHA256 {expected})' if expected else '') + '.'
            ) from error
    actual = hashlib.sha256(target.read_bytes()).hexdigest()
    if expected and actual != expected:
        raise RuntimeError(f'Backbone checksum differs from the training record: {actual} vs {expected}. '
                           'Use the original ImageNet ResNet18 file, or compare with the saved formal results only.')
    print('ImageNet backbone ready and verified:', target.name, '| torch', torch.__version__, '| torchvision', torchvision.__version__)

ensure_imagenet_backbone()

## 2. Data, identity linkage and leakage checks
The raw subset has 312 images / 78 candidate pairs; pair 81 remains excluded by the original Handiprint eligibility decision. The retained 308 inputs represent 77 pairs. All sides, full/partial conditions and media from one pair retain one split. Identity-inferred pairs are reported, not silently removed. Pair 157 remains Nike according to metadata; suffix discrepancies are not resolved by visually relabelling its tread.

Exact hash agreement establishes artifact continuity, not semantic crop quality or absence of near duplicates. Seven mappings (9,70,76,79,114,120,158) remain metadata-inferred. The original and blood test cohorts share shoe pairs and prior exposure.


In [ ]:
def file_hash(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as handle:
        for block in iter(lambda:handle.read(1024*1024),b''):
            h.update(block)
    return h.hexdigest()

lock=json.loads((BASE/'preprocessing_lock.json').read_text())
for name,field in [('crop_annotations.csv','training_annotations_sha256'),('blood_manifest_audit.csv','audit_sha256'),('original_split_manifest.csv','split_sha256')]:
    assert file_hash(BASE/name)==lock[field],name
manifest=pd.read_csv(GPU/'dataset_manifest.csv',keep_default_na=False)
audit_table=pd.read_csv(BASE/'blood_manifest_audit.csv',keep_default_na=False)
annotations=pd.read_csv(BASE/'crop_annotations.csv',keep_default_na=False)
assert len(audit_table)==312 and len(manifest)==308 and manifest.key.is_unique
assert manifest.groupby('pair_id').split.nunique().eq(1).all()
assert not manifest.pair_id.eq(81).any()
assert manifest.groupby('pair_id').y.nunique().eq(1).all()
assert manifest.groupby('crop_sha256').split.nunique().eq(1).all()
assert annotations.key.is_unique and len(annotations)==180 and annotations.status.eq('approved').all()
assert set(annotations.key)==set(manifest.loc[manifest.split.eq('train'),'key'])
print('Original lock verified; 180 approved training records retained.')
display(manifest.groupby(['split','brand','condition']).agg(images=('key','size'),pairs=('pair_id','nunique')))
identity=manifest[['pair_id','split','brand']].drop_duplicates()
identity['identity_basis']=np.where(identity.pair_id.isin([9,70,76,79,114,120,158]),'metadata_inferred','Image-info_supported')
display(identity.groupby(['split','identity_basis']).size().rename('pairs'))
display(audit_table.groupby(['brand','Style','Size (US)']).pair_id.nunique().rename('pairs'))

if BUNDLE_DIR:
    for row in manifest.itertuples():
        image_path=BUNDLE_DIR/'input/inputs'/row.split/f'{row.key}.png'
        assert image_path.is_file(),f'Incomplete portable package: {image_path}'
        assert file_hash(image_path)==row.crop_sha256,row.key
    print('All 308 bundled RGB224 inputs verified without Google Drive.')


## 3. Frozen preprocessing and training-only examples
Automatic red localization was a candidate pilot and was not adopted. The finalized procedure retains faint/disconnected regions using reviewed RGB rectangles; it preserves aspect ratio and native orientation and applies no perspective warp. Full-resolution training ROIs and externally cropped held-out inputs use the same frozen RGB224 canvas conversion. These interfaces can still induce distribution differences. Grayscale is a controlled model input change on identical RGB crops, not a new crop.

REBUILD_INPUTS=False prevents repeated import/export during normal execution. The original importer is retained below for documented reproduction, and only runs on explicit request; it verifies/reuses existing prepared artifacts. No original lock is recreated.


In [ ]:
PIPELINE = r"""
def get_crop(path,bounds):
    with Image.open(path) as image:
        rgb = ImageOps.exif_transpose(image).convert('RGB')
    x0,y0,x1,y1 = map(float,bounds)
    if not (0<=x0<x1<=1 and 0<=y0<y1<=1):
        raise ValueError('Invalid crop bounds')
    box = (
        math.floor(x0*rgb.width), math.floor(y0*rgb.height),
        math.ceil(x1*rgb.width), math.ceil(y1*rgb.height)
    )
    crop = rgb.crop(box)
    fitted = ImageOps.contain(
        crop,(224,224),Image.Resampling.LANCZOS
    )
    canvas = Image.new('RGB',(224,224),'white')
    canvas.paste(
        fitted,((224-fitted.width)//2,(224-fitted.height)//2)
    )
    return rgb,box,crop,canvas
"""
assert hashlib.sha256(PIPELINE.encode()).hexdigest()==lock['pipeline_sha256']
exec(PIPELINE,globals())
print('Frozen ROI/canvas source hash verified.')


In [ ]:
if REBUILD_INPUTS:
    assert BUNDLE_DIR is None, 'Rebuilding requires original Drive inputs.'
    from pathlib import Path
    import re
    import json
    import math
    import hashlib
    import textwrap

    import pandas as pd
    from PIL import Image, ImageOps
    import sys

    if not Path("/content/drive/MyDrive").is_dir():
        try:
            from google.colab import drive
            drive.mount("/content/drive")
        except ImportError:
            pass

    ROOT = Path(globals().get("ROOT", "/content/drive/MyDrive/FOOTWEAR_PRIVATE_PROJECT"))
    BASE = Path(globals().get("CANONICAL_RUN", ROOT / "results/blood_tile/20261025T183821Z_90054831"))
    INCOMING = ROOT / "data/blood_tile_crops_external"
    OUT = ROOT / "results/blood_external_v1"
    OUT.mkdir(parents=True, exist_ok=True)


    def file_sha(path):
        digest = hashlib.sha256()
        with Path(path).open("rb") as handle:
            for block in iter(lambda: handle.read(1024 * 1024), b""):
                digest.update(block)
        return digest.hexdigest()


    def filename_key(path):
        value = Path(path).stem

        # Handle names such as original.JPG.png.
        if Path(value).suffix.lower() in {".jpg", ".jpeg", ".png"}:
            value = Path(value).stem

        # Ignore only the numbered export prefix.
        value = re.sub(
            r"^[0-9]+_(?=[0-9]{3}[LR]B[FP]T)",
            "", value, flags=re.I
        )
        return value.casefold()


    def import_desktop_crops():
        assert INCOMING.is_dir(), f"Folder not found: {INCOMING}"

        # Verify the original locked training artifacts.
        lock = json.loads((BASE / "preprocessing_lock.json").read_text())

        for name, field in [
            ("crop_annotations.csv", "training_annotations_sha256"),
            ("blood_manifest_audit.csv", "audit_sha256"),
            ("original_split_manifest.csv", "split_sha256"),
        ]:
            assert file_sha(BASE / name) == lock[field], (
                f"Locked input changed: {name}"
            )

        # Reuse a completed import without generating duplicate directories.
        import_lock = OUT / "external_import_lock.json"
        if import_lock.exists():
            saved = json.loads(import_lock.read_text())
            assert saved["ready"]
            assert saved["original_preprocessing_lock_sha256"] == file_sha(
                BASE / "preprocessing_lock.json"
            )
            assert file_sha(OUT / "dataset_manifest.csv") == saved[
                "dataset_manifest_sha256"
            ]
            assert file_sha(OUT / "external_crop_import.csv") == saved[
                "external_import_sha256"
            ]

            cached = pd.read_csv(
                OUT / "dataset_manifest.csv", keep_default_na=False
            )
            for row in cached[cached.crop_path.ne("")].itertuples():
                assert file_sha(row.crop_path) == row.crop_sha256

            print("Existing import verified and reused.")
            print(cached.groupby(["split", "status"]).size())
            return cached

        assert not (OUT / "experiment_plan.json").exists(), (
            "Do not replace model inputs after training begins."
        )

        # Recover the hash-matched crop/canvas implementation.
        source_path = (
            ROOT / "results/blood_completion_v1/"
            "locked_inputs/frozen_crop_pipeline_source.txt"
        )
        frozen_source = (
            source_path.read_text()
            if source_path.exists()
            else globals().get("PIPELINE")
        )
        assert isinstance(frozen_source, str), (
            "Frozen crop source not found. Run the original crop-function "
            "definition cell first; do not Run all."
        )
        assert hashlib.sha256(frozen_source.encode()).hexdigest() == lock[
            "pipeline_sha256"
        ]

        namespace = {"Image": Image, "ImageOps": ImageOps, "math": math}
        exec(textwrap.dedent(frozen_source), namespace)
        crop_function = namespace["get_crop"]

        audit = pd.read_csv(BASE / "blood_manifest_audit.csv")
        training = pd.read_csv(
            BASE / "crop_annotations.csv", keep_default_na=False
        )
        expected = audit[audit["split"].isin(["val", "test"])].copy()

        assert len(training) == 180 and len(expected) == 128
        assert training.key.is_unique
        assert training.status.eq("approved").all()
        assert training.checks_passed.astype(str).str.lower().eq("true").all()
        assert set(training.key) == set(
            audit.loc[audit["split"].eq("train"), "key"]
        )

        expected["file_key"] = expected.path.map(filename_key)
        assert expected.file_key.is_unique

        # Locate supported image files, ignoring hidden system files.
        files_by_key = {}
        for path in sorted(INCOMING.rglob("*")):
            if (
                path.is_file()
                and not path.name.startswith(".")
                and "__MACOSX" not in path.parts
                and path.suffix.lower()
                in {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"}
            ):
                files_by_key.setdefault(filename_key(path), []).append(path)

        expected_keys = set(expected.file_key)
        issues = [
            {"file": str(path), "issue": "Filename not in expected records"}
            for key, paths in files_by_key.items()
            if key not in expected_keys
            for path in paths
        ]

        imported = []

        for row in expected.sort_values("key").itertuples():
            record = {
                "key": row.key,
                "pair_id": int(row.pair_id),
                "side": row.side,
                "condition": row.condition,
                "brand": row.brand,
                "y": 0 if row.brand == "Adidas" else 1,
                "split": row.split,
                "image_sha256": row.sha256,
                "original_path": row.path,
                "annotation_method": "desktop_manual_crop",
                "status": "needs_fix",
                "notes": "",
                "external_path": "",
            }

            try:
                candidates = files_by_key.get(row.file_key, [])
                assert len(candidates) == 1, (
                    "Missing crop" if not candidates
                    else "Multiple files match this image"
                )

                path = candidates[0]
                folder_splits = {
                    part.lower()
                    for part in path.relative_to(INCOMING).parent.parts
                    if part.lower() in {"val", "test"}
                }
                assert not folder_splits or folder_splits == {row.split}, (
                    "Folder split disagrees with the original split"
                )

                source = Path(row.path)
                if not source.exists():
                    source = (
                        ROOT / "data/blood_tile_jpg"
                        / row.folder_brand / source.name
                    )
                assert file_sha(source) == row.sha256, (
                    "Original source bytes changed"
                )

                with Image.open(source) as image:
                    original_size = image.size
                    if image.getexif().get(274, 1) in {5, 6, 7, 8}:
                        original_size = original_size[::-1]

                with Image.open(path) as image:
                    image.load()
                    image = ImageOps.exif_transpose(image)

                    assert image.mode in {"RGB", "RGBA"}, (
                        "An RGB colour crop is required; "
                        "grayscale/palette export detected"
                    )
                    if image.mode == "RGBA":
                        assert image.getchannel("A").getextrema() == (255, 255), (
                            "Transparent crop detected"
                        )
                    width, height = image.size

                assert min(width, height) >= 16, "Crop is too small"
                assert (
                    width <= original_size[0]
                    and height <= original_size[1]
                ), "Crop exceeds source dimensions; check rotation or resizing"
                assert (width, height) != original_size, (
                    "This appears to be an uncropped full image"
                )

                record.update(
                    status="external_crop_imported",
                    external_path=str(path),
                    external_sha256=file_sha(path),
                    width=width,
                    height=height,
                )

            except Exception as error:
                record["notes"] = str(error)
                issues.append({
                    "file": Path(row.path).name,
                    "issue": str(error),
                })

            imported.append(record)

        imported_table = pd.DataFrame(imported)
        imported_table.to_csv(OUT / "external_crop_import.csv", index=False)
        pd.DataFrame(issues, columns=["file", "issue"]).to_csv(
            OUT / "import_issues.csv", index=False
        )

        if issues:
            print("Files need attention; no training has run.")
            display(pd.DataFrame(issues))
            print("Issue log:", OUT / "import_issues.csv")
            return None

        # Generate training inputs from the existing approved coordinates.
        cache = OUT / "crops"
        cache.mkdir(exist_ok=True)
        dataset = []
        indexed = audit.set_index("key")

        for number, review in enumerate(training.itertuples(), 1):
            row = indexed.loc[review.key]
            source = Path(row.path)
            if not source.exists():
                source = (
                    ROOT / "data/blood_tile_jpg"
                    / row.folder_brand / source.name
                )

            assert file_sha(source) == review.image_sha256 == row.sha256
            _, _, _, canvas = crop_function(
                source, json.loads(review.bounds)
            )
            assert canvas.mode == "RGB" and canvas.size == (224, 224)

            target = cache / f"{review.key}.png"
            canvas.save(target)

            dataset.append({
                "key": review.key,
                "pair_id": int(row.pair_id),
                "side": row.side,
                "condition": row.condition,
                "brand": row.brand,
                "y": 0 if row.brand == "Adidas" else 1,
                "split": "train",
                "status": "approved",
                "annotation_method": "training_manual_roi",
                "image_sha256": row.sha256,
                "crop_path": str(target),
                "crop_sha256": file_sha(target),
                "bounds": review.bounds,
                "notes": "",
            })

            if number % 30 == 0:
                print(f"Training inputs prepared: {number}/180", flush=True)

        # Apply the same RGB resize/padding implementation to desktop crops.
        for number, record in enumerate(imported, 1):
            _, _, _, canvas = crop_function(
                record["external_path"], [0, 0, 1, 1]
            )
            assert canvas.mode == "RGB" and canvas.size == (224, 224)

            target = cache / f"{record['key']}.png"
            canvas.save(target)

            dataset.append({
                **record,
                "crop_path": str(target),
                "crop_sha256": file_sha(target),
                "bounds": "",
            })

            if number % 32 == 0:
                print(f"Held-out inputs prepared: {number}/128", flush=True)

        result = pd.DataFrame(dataset)
        assert len(result) == 308 and result.key.is_unique
        assert result.groupby("pair_id")["split"].nunique().eq(1).all()
        assert not result.crop_sha256.duplicated().any(), (
            "Duplicate processed images found; review before training"
        )

        result.to_csv(OUT / "dataset_manifest.csv", index=False)

        provenance = {
            "ready": True,
            "annotation_method": "desktop_manual_crop",
            "training_images": 180,
            "validation_images": 64,
            "test_images": 64,
            "original_preprocessing_lock_sha256": file_sha(
                BASE / "preprocessing_lock.json"
            ),
            "frozen_pipeline_sha256": lock["pipeline_sha256"],
            "dataset_manifest_sha256": file_sha(
                OUT / "dataset_manifest.csv"
            ),
            "external_import_sha256": file_sha(
                OUT / "external_crop_import.csv"
            ),
            "pixel_source": (
                "Training: original JPG with locked ROI. "
                "Held-out: user-provided desktop crop followed by "
                "the frozen RGB canvas conversion."
            ),
            "limitation": (
                "File checks do not independently verify faint-tread "
                "preservation or exact source-pixel correspondence. "
                "No checkbox approvals are fabricated."
            ),
        }
        (OUT / "external_import_lock.json").write_text(
            json.dumps(provenance, indent=2)
        )

        print("\nImport completed:")
        print(result.groupby(["split", "status"]).size())
        print("\nOutput folder:", OUT)
        return result


    dataset_manifest = import_desktop_crops()
else:
    print('Prepared inputs reused; no reviewer or acquisition launched.')


In [ ]:
if SHOW_TRAIN_PREVIEWS:
    import matplotlib.pyplot as plt
    selected=[]
    for brand in ['Adidas','Nike']:
        eligible=manifest[manifest.split.eq('train')&manifest.brand.eq(brand)]
        pair_ids=sorted(eligible.pair_id.unique())[:4]
        for condition in ['Full','Partial']:
            for pair_id in pair_ids:
                row=eligible[eligible.pair_id.eq(pair_id)&eligible.condition.eq(condition)].sort_values('side').iloc[0]
                selected.append(row)
    preview_path=lambda row:(BUNDLE_DIR/'input/inputs'/row.split/f'{row.key}.png') if BUNDLE_DIR else Path(row.crop_path)
    if all(preview_path(row).is_file() for row in selected):
        fig,axes=plt.subplots(4,4,figsize=(9,10))
        for ax,row in zip(axes.ravel(),selected):
            assert file_hash(preview_path(row))==row.crop_sha256
            with Image.open(preview_path(row)) as image:ax.imshow(image)
            ax.set_title(f'{row.brand} / pair {row.pair_id} / {row.condition}',fontsize=8);ax.axis('off')
        fig.suptitle('Four different training pairs per class and condition; no held-out crop selection')
        fig.tight_layout();plt.show()
    else:
        print('Training canvases are not in this analysis-only bundle. Previews require the documented Drive inputs; no substitute images were used.')


### Review-history reconciliation
The latest run contains only the final 12 keys because the earlier 168 were resumed from an older run. The historical packages were uploaded and merged for this reconciliation. Embedded original history events preserve the evidence; this checks record continuity rather than independently certifying the semantics of every crop. Aggregate approval is not a substitute for manufacturing missing individual checkbox fields.


In [ ]:
HISTORY_ARCHIVE_HASHES={'crop_review_handoff_20261024T232353072257Z.zip': '2acc19a6f2004db74a973d991b10d7e25adc165f1bcc4ba22df1a7a71be4c4a8', 'crop_review_handoff_20261025T190536040405Z (1).zip': '48870176a707728f90f965cf22a4d352242ee3e132b243611a1574a4a14bd617'}
history_paths=(list((BUNDLE_DIR/'histories').glob('*.zip')) if BUNDLE_DIR else [
    ROOT/'results/blood_tile/20261024T185403Z_ba785dda/crop_review_handoff_20261024T232353072257Z.zip',
    ROOT/'results/blood_tile/20261025T183821Z_90054831/crop_review_handoff_20261025T190536040405Z.zip'])
expected_hashes=set(HISTORY_ARCHIVE_HASHES.values())
events=[]
for path in history_paths:
    if path.exists():
        assert file_hash(path) in expected_hashes,path.name
        with zipfile.ZipFile(path) as archive:
            for name in archive.namelist():
                if name.endswith('crop_annotation_history.jsonl'):
                    for line in archive.read(name).decode().splitlines():
                        if line.strip():
                            event=json.loads(line);event['history_source']=path.name;events.append(event)
latest={}
for event in sorted(events,key=lambda event:event.get('reviewed_at_utc','')):
    latest[event['key']]=event
if set(annotations.key).issubset(latest):
    reconciliation=[]
    for row in annotations.itertuples():
        event=latest[row.key]
        assert np.allclose(json.loads(row.bounds),event['bounds'],rtol=0,atol=0),row.key
        assert row.status==event['status'] and row.image_sha256==event['image_sha256'],row.key
        assert str(row.checks_passed).lower()==str(event['checks_passed']).lower(),row.key
        reconciliation.append({'key':row.key,'history_source':event['history_source'],'latest_record_matches':True})
    pd.DataFrame(reconciliation).to_csv(FINAL_OUTPUT/'history_reconciliation.csv',index=False)
    print('Matched history for all 180 training keys. No re-review required.')
    display(pd.DataFrame(reconciliation).groupby('history_source').size().rename('keys'))
else:
    print('Historical source ZIPs are not both available here. Reference analysis verified all 180 keys from the uploaded archives; this run has not repeated that check. Missing local archives do not require image re-review.')
    (FINAL_OUTPUT/'historical_reference_summary.json').write_text(json.dumps({'reference_keys_verified':180,'source_archive_hashes':HISTORY_ARCHIVE_HASHES,'repeated_in_this_run':False},indent=2))


## 4. Completed Handiprint phase: archived evidence
The original phase trained frozen-backbone ResNet18 heads and a HOG classifier on Handiprint images. Its main mask analysis had a ceiling effect. The severe-mask extension was developed after original test exposure and is exploratory. At 80% crop-area masking, the supplied report records BA 53.75% for the original CNN versus 95.00% for occlusion-augmented training: +41.25 percentage points, paired interval [35.63,46.88]. This is archived evidence from the supplied report/extension notebook, not recomputation on unavailable original TIFFs.

The original augmentation checkpoint was not included in results_to_share.zip. Consequently the cross-medium comparator below is the **original** Handiprint head. This notebook does not claim to test transfer of the earlier augmented head. Original 80% crop-area masking and the blood experiment's padded-canvas masking are different severity definitions.


In [ ]:
assert LEGACY_ARCHIVE.is_file(), f'Supply the existing archive: {LEGACY_ARCHIVE}'
formal_plan=json.loads((GPU/'experiment_plan.json').read_text())
assert file_hash(LEGACY_ARCHIVE)==formal_plan['legacy_archive_sha256']
with zipfile.ZipFile(LEGACY_ARCHIVE) as archive:
    legacy_summary=json.loads(archive.read('results/results_summary.json'))
    original_metrics=pd.read_csv(io.BytesIO(archive.read('results/tables/metrics_image_level.csv')))
    original_plan=json.loads(archive.read('results/plan_lock.json'))
assert original_plan['split_manifest_sha256']==lock['split_sha256']
print('Archived Handiprint unmodified test results:')
display(original_metrics[original_metrics.condition.eq('unmodified')&original_metrics.metric.eq('balanced_accuracy')])
print('Archived severe-mask extension at 80% crop-area masking:')
display(pd.DataFrame([{'model':'Original Handiprint CNN','balanced_accuracy':.5375},{'model':'Occlusion-augmented Handiprint CNN','balanced_accuracy':.95}]))

archived_severe=pd.DataFrame([
    [0,100,100,0,0,0],[50,92.50,100,7.50,4.38,11.25],
    [60,79.38,100,20.63,15.63,25.63],[70,66.25,98.75,32.50,26.88,38.13],
    [80,53.75,95.00,41.25,35.63,46.88]],
    columns=['mask_percent','original_ba_percent','augmented_ba_percent','gain_pp','gain_ci_low_pp','gain_ci_high_pp'])
print('Archived Handiprint severe-mask results, as rounded in the supplied original report:')
display(archived_severe)
(FINAL_OUTPUT/'tables').mkdir(exist_ok=True)
archived_severe.to_csv(FINAL_OUTPUT/'tables/archived_handiprint_severe.csv',index=False)


## 5. Blood model definitions, training and GPU provenance
The model is an ImageNet ResNet18 feature extractor with a newly trained linear binary head (513 trainable parameters). The backbone stays frozen, including batch-normalization statistics. Adam, class-weighted BCE-with-logits, batch 16, weight decay 0.0001, up to 40 epochs, patience 7, and validation improvement threshold 0.0001 are used.

RGB tests learning rates 0.001 and 0.01 across three seeds. The selected RGB rate is reused for grayscale and RGB-occlusion. Three seed probabilities are averaged at threshold 0.5. Augmentation uses modest affine/brightness/contrast transforms; occlusion probability is 0.5 and mask area is 10-70% of the entire white-padded canvas, not tread area. HOG C is selected by validation BA with log-loss as tie-breaker.

Only blood_external_gpu_v1 is formal. The earlier CPU/mixed folder is preserved, never compared to choose the better score. No random-initialized full ResNet or 384-pixel experiment is claimed. The original GPU logs remain in the preceding saved training notebook; this submission verifies all twelve receipt hashes.


In [ ]:
# Training and validation selection; no test predictions are computed here.
from pathlib import Path
import os,io,json,hashlib,random,time,platform,zipfile
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG',':4096:8')
import numpy as np,pandas as pd
from PIL import Image
import torch,torch.nn as nn,torchvision,joblib
from torchvision.transforms import functional as TF
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss,balanced_accuracy_score
from skimage.feature import hog


def checksum(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as f:
        for part in iter(lambda:f.read(1024*1024),b''):h.update(part)
    return h.hexdigest()


def json_save(data,path):
    path=Path(path);temporary=path.with_suffix(path.suffix+'.tmp')
    temporary.write_text(json.dumps(data,indent=2,default=str));temporary.replace(path)


CONFIG={'variants':['rgb','gray','rgb_occlusion'],'seeds':[0,1,2],'lr_grid':[.001,.01],
        'epochs':40,'patience':7,'batch_size':16,'weight_decay':.0001,'hog_C':[.01,.1,1.,10.],
        'backbone':'ResNet18_IMAGENET1K_V1','checkpoint':'clean validation conditional log-loss',
        'threshold':.5,'occlusion_probability':.5,'occlusion_area_range':[.1,.7],
        'mask_area_definition':'Full 224x224 canvas, including padding; not tread area',
        'learning_rate_rule':'Select RGB learning rate on validation, then reuse it for gray and rgb_occlusion',
        'bootstrap_pairs':2000,'bootstrap_seed':673,'test_is_exploratory':True}


class FrozenCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone=torchvision.models.resnet18(weights=torchvision.models.ResNet18_Weights.IMAGENET1K_V1)
        self.backbone.fc=nn.Identity()
        for parameter in self.backbone.parameters():parameter.requires_grad=False
        self.head=nn.Linear(512,1)
        self.register_buffer('mean',torch.tensor([.485,.456,.406]).view(1,3,1,1))
        self.register_buffer('std',torch.tensor([.229,.224,.225]).view(1,3,1,1))
    def train(self,mode=True):
        super().train(mode);self.backbone.eval();return self
    def forward(self,x):
        return self.head(self.backbone((x-self.mean)/self.std)).squeeze(1)


def cnn_input(images,variant,device):
    if variant=='gray':
        images=np.stack([np.repeat(np.asarray(Image.fromarray(image).convert('L'))[:,:,None],3,axis=2) for image in images])
    return torch.from_numpy((images.astype(np.float32)/255).transpose(0,3,1,2).copy()).to(device)


def cnn_scores(model,images,variant):
    model.eval();values=[];device=next(model.parameters()).device
    with torch.no_grad():
        for start in range(0,len(images),32):
            values.extend(torch.sigmoid(model(cnn_input(images[start:start+32],variant,device))).cpu().numpy())
    return np.asarray(values)


def augment(images,variant,config):
    output=[]
    for image in images:
        image=image.clone()
        if variant=='rgb_occlusion' and np.random.rand()<config['occlusion_probability']:
            area=np.random.uniform(*config['occlusion_area_range'])
            rh=np.exp(np.random.uniform(np.log(area),0));rw=area/rh
            height,width=image.shape[-2:];mh=max(1,round(rh*height));mw=max(1,round(rw*width))
            top=np.random.randint(height-mh+1);left=np.random.randint(width-mw+1)
            image[:,top:top+mh,left:left+mw]=1
        image=TF.affine(image,angle=float(np.random.uniform(-5,5)),
                        translate=[int(np.random.uniform(-.05,.05)*224) for _ in range(2)],
                        scale=float(np.random.uniform(.95,1.05)),shear=[0.,0.],fill=1.)
        image=TF.adjust_contrast(image,float(np.random.uniform(.9,1.1)))
        image=TF.adjust_brightness(image,float(np.random.uniform(.95,1.05)))
        output.append(image.clamp(0,1))
    return torch.stack(output)


def fit_head(x,y,xval,yval,variant,lr,seed,config,device):
    random.seed(seed);np.random.seed(seed);torch.manual_seed(seed)
    if torch.cuda.is_available():torch.cuda.manual_seed_all(seed)
    model=FrozenCNN().to(device)
    optimizer=torch.optim.Adam(model.head.parameters(),lr=lr,weight_decay=config['weight_decay'])
    loss_function=nn.BCEWithLogitsLoss(pos_weight=torch.tensor(float(np.sum(y==0)/np.sum(y==1)),device=device))
    best=np.inf;state=None;bad=0;history=[]
    for epoch in range(config['epochs']):
        model.train();order=np.random.permutation(len(x));total=0.
        for start in range(0,len(x),config['batch_size']):
            ids=order[start:start+config['batch_size']]
            xb=augment(cnn_input(x[ids],variant,device),variant,config)
            target=torch.tensor(y[ids],dtype=torch.float32,device=device)
            optimizer.zero_grad();loss=loss_function(model(xb),target);loss.backward();optimizer.step()
            total+=float(loss.item())*len(ids)
        score=cnn_scores(model,xval,variant)
        value=float(log_loss(yval,np.clip(score,1e-7,1-1e-7),labels=[0,1]))
        history.append({'variant':variant,'lr':lr,'seed':seed,'epoch':epoch+1,'train_loss':total/len(x),'val_logloss':value})
        if epoch==0 or (epoch+1)%5==0:print(f'  epoch {epoch+1}: validation log-loss {value:.5f}',flush=True)
        if value<best-1e-4:
            best=value;state={k:v.detach().cpu().clone() for k,v in model.head.state_dict().items()};bad=0
        else:
            bad+=1
            if bad>=config['patience']:break
    assert state is not None
    model.head.load_state_dict(state)
    return state,pd.DataFrame(history),best


def hog_matrix(images):
    return np.stack([hog(np.asarray(Image.fromarray(image).convert('L').resize((128,128)))/255,
                         orientations=9,pixels_per_cell=(16,16),cells_per_block=(2,2),block_norm='L2-Hys') for image in images])

def train_project(root,base,config=None,experiment="blood_external_gpu_v1",input_resolver=None,legacy_archive=None):
    config=dict(CONFIG if config is None else config);root,base=Path(root),Path(base)
    resolve_input=input_resolver if input_resolver is not None else lambda row:Path(row.crop_path)
    work=root/'results'/experiment
    assert work.is_dir(),'Run the desktop crop import first.'
    for folder in ['models','tables','figures','report']:(work/folder).mkdir(exist_ok=True)
    record=json.loads((work/'external_import_lock.json').read_text())
    assert record['ready'] and checksum(work/'dataset_manifest.csv')==record['dataset_manifest_sha256']
    assert checksum(work/'external_crop_import.csv')==record['external_import_sha256']
    assert checksum(base/'preprocessing_lock.json')==record['original_preprocessing_lock_sha256']
    data=pd.read_csv(work/'dataset_manifest.csv',keep_default_na=False)
    assert len(data)==308 and data.key.is_unique and data.groupby('pair_id')['split'].nunique().eq(1).all()
    assert data.groupby('split').size().to_dict()=={'test':64,'train':180,'val':64}
    for row in data[data['split'].isin(['train','val']) & data.crop_path.ne('')].itertuples():
        assert checksum(resolve_input(row))==row.crop_sha256,'Cached input changed: '+row.key
    early_selection=work/'model_selection.json'
    if early_selection.exists():
        frozen_plan=json.loads((work/'experiment_plan.json').read_text())
        assert frozen_plan['config']==config
        assert frozen_plan['manifest_sha256']==record['dataset_manifest_sha256']
        assert frozen_plan['heldout_annotations_sha256']==record['external_import_sha256']
        assert frozen_plan['crop_lock_sha256']==record['original_preprocessing_lock_sha256']
        saved=json.loads(early_selection.read_text())
        for name,value in saved['model_sha256'].items():assert checksum(work/'models'/name)==value
        print('Saved selected models verified and reused. No retraining.');return saved
    def arrays(split):
        rows=data[data['split'].eq(split)&data.crop_path.ne('')].sort_values('key')
        assert set(rows.y)=={0,1}
        images=[]
        for row in rows.itertuples():
            with Image.open(resolve_input(row)) as image:
                assert image.size==(224,224) and image.mode=='RGB'
                images.append(np.asarray(image))
        return rows,np.stack(images),rows.y.to_numpy(dtype=int)
    tr,x,y=arrays('train');va,xval,yval=arrays('val')
    legacy=Path(legacy_archive) if legacy_archive is not None else root/'results_to_share.zip'
    assert legacy.exists(),'Upload your existing results_to_share.zip to '+str(legacy)+' for the fixed Handiprint transfer comparator.'
    with zipfile.ZipFile(legacy) as archive:
        old_plan=json.loads(archive.read('results/plan_lock.json'))
        assert old_plan['split_manifest_sha256']==json.loads((base/'preprocessing_lock.json').read_text())['split_sha256']
        for seed in [0,1,2]:
            name=f'resnet18_selected_frozen_backbone_seed{seed}.pt'
            raw=archive.read('results/models/'+name)
            assert hashlib.sha256(raw).hexdigest()==old_plan['model_weight_sha256'][name]
    plan={'config':config,'manifest_sha256':checksum(work/'dataset_manifest.csv'),
          'heldout_annotations_sha256':checksum(work/'external_crop_import.csv'),
          'crop_lock_sha256':checksum(base/'preprocessing_lock.json'),
          'legacy_archive_sha256':checksum(legacy),'implementation_id':'blood_external_frozen_cnn_v1'}
    plan_path=work/'experiment_plan.json'
    if plan_path.exists():
        assert json.loads(plan_path.read_text())==plan,'Settings or inputs changed; use a separate experiment.'
    else:json_save(plan,plan_path)
    selection_path=work/'model_selection.json'
    if selection_path.exists():
        saved=json.loads(selection_path.read_text())
        for name,value in saved['model_sha256'].items():assert checksum(work/'models'/name)==value
        print('Saved selected models verified and reused. No retraining.');return saved
    assert not (work/'evaluation_lock.json').exists(),'Preserve the evaluated experiment; do not retrain it.'
    torch.set_num_threads(min(4,os.cpu_count() or 1));torch.backends.cudnn.benchmark=False;torch.backends.cudnn.deterministic=True
    torch.use_deterministic_algorithms(True)
    device='cuda' if torch.cuda.is_available() else 'cpu'
    print('Device:',device,'| training:',len(tr),'| validation:',len(va),'| test not evaluated',flush=True)
    # Verify and copy the original heads without retraining the Handiprint study.
    legacy_heads=[]
    with zipfile.ZipFile(legacy) as archive:
        prior=json.loads(archive.read('results/plan_lock.json'))
        assert prior['split_manifest_sha256']==json.loads((base/'preprocessing_lock.json').read_text())['split_sha256']
        for seed in [0,1,2]:
            name=f'resnet18_selected_frozen_backbone_seed{seed}.pt';raw=archive.read('results/models/'+name)
            assert hashlib.sha256(raw).hexdigest()==prior['model_weight_sha256'][name]
            state=torch.load(io.BytesIO(raw),map_location='cpu',weights_only=True)
            state={key.replace('head.',''):value for key,value in state.items() if key.startswith('head.')}
            assert set(state)=={'weight','bias'} and tuple(state['weight'].shape)==(1,512)
            target=work/'models'/f'legacy_handiprint_seed{seed}.pt';torch.save(state,target);legacy_heads.append(target.name)
    # HOG hyperparameters use validation data only.
    fx,fv=hog_matrix(x),hog_matrix(xval);hog_runs=[];candidates={}
    for c in config['hog_C']:
        model=make_pipeline(StandardScaler(),LogisticRegression(C=c,class_weight='balanced',max_iter=3000,random_state=42))
        model.fit(fx,y);scores=model.predict_proba(fv)[:,1]
        hog_runs.append({'C':c,'val_balanced_accuracy':float(balanced_accuracy_score(yval,scores>.5)),
                         'val_logloss':float(log_loss(yval,np.clip(scores,1e-7,1-1e-7),labels=[0,1]))})
        candidates[c]=model
    grid=pd.DataFrame(hog_runs).sort_values(['val_balanced_accuracy','val_logloss'],ascending=[False,True])
    best_c=float(grid.iloc[0].C);joblib.dump(candidates[best_c],work/'models/hog.joblib');grid.to_csv(work/'tables/hog_validation.csv',index=False)
    selection={'majority_score':float(y.mean()),'hog_C':best_c,'variants':{},'background_available':False,
               'legacy_available':True,'legacy_heads':legacy_heads,'coverage_validation':float(len(va)/64)}
    runs=[];histories=[]
    assert config['variants'][0]=='rgb','RGB must run first for the shared learning-rate rule.'
    for variant in config['variants']:
        variant_runs=[]
        learning_rates=config['lr_grid'] if variant=='rgb' else [selection['variants']['rgb']['lr']]
        for lr in learning_rates:
            for seed in config['seeds']:
                target=work/'models'/f'{variant}_lr{lr:g}_seed{seed}.pt';receipt=target.with_suffix('.json')
                if target.exists() and receipt.exists():
                    saved=json.loads(receipt.read_text());assert checksum(target)==saved['sha256']
                    value=saved['val_loss'];history=pd.read_csv(target.with_suffix('.csv'))
                    print('Reused completed fit:',variant,lr,seed,flush=True)
                else:
                    print('Training:',variant,'lr=',lr,'seed=',seed,flush=True)
                    state,history,value=fit_head(x,y,xval,yval,variant,lr,seed,config,device)
                    torch.save(state,target);history.to_csv(target.with_suffix('.csv'),index=False)
                    json_save({'sha256':checksum(target),'val_loss':value,'device':device,'torch':torch.__version__,
                               'variant':variant,'lr':lr,'seed':seed},receipt)
                    if torch.cuda.is_available():torch.cuda.empty_cache()
                histories.append(history);variant_runs.append({'variant':variant,'lr':lr,'seed':seed,'val_loss':value})
        table=pd.DataFrame(variant_runs);chosen=float(table.groupby('lr').val_loss.mean().idxmin())
        selection['variants'][variant]={'lr':chosen,'heads':[f'{variant}_lr{chosen:g}_seed{seed}.pt' for seed in config['seeds']]}
        runs.extend(variant_runs)
        print('Selected learning rate:',variant,chosen,flush=True)
    pd.DataFrame(runs).to_csv(work/'tables/cnn_validation.csv',index=False)
    pd.concat(histories,ignore_index=True).to_csv(work/'tables/cnn_learning_curves.csv',index=False)
    names=['hog.joblib']+legacy_heads+[name for variant in selection['variants'].values() for name in variant['heads']]
    selection['fit_provenance']={name:json.loads((work/'models'/name).with_suffix('.json').read_text())
                                 for value in selection['variants'].values() for name in value['heads']}
    selection['model_sha256']={name:checksum(work/'models'/name) for name in names}
    json_save(selection,selection_path)
    from importlib.metadata import version
    backbone_file=Path(torch.hub.get_dir())/'checkpoints/resnet18-f37072fd.pth'
    json_save({'backbone_weights_sha256':checksum(backbone_file) if backbone_file.exists() else None,
               'python':platform.python_version(),'device':device,'packages':{name:version(name) for name in
              ['numpy','pandas','pillow','torch','torchvision','scikit-learn','scikit-image','joblib']}},work/'environment.json')
    print('TRAINING COMPLETE. Models selected using validation data. Test metrics have not been computed.')
    return selection


In [ ]:
# Formal GPU artifacts remain read-only. Optional fits go to another directory.
if RUN_TRAINING:
    import shutil
    reproduction_root=BUNDLE_DIR if BUNDLE_DIR else ROOT
    reproduction_name='blood_reproduction_v1'
    reproduction_dir=reproduction_root/'results'/reproduction_name
    reproduction_dir.mkdir(parents=True,exist_ok=True)
    for name in ['dataset_manifest.csv','external_crop_import.csv','external_import_lock.json']:
        source=GPU/name;destination=reproduction_dir/name
        if destination.exists():assert file_hash(destination)==file_hash(source),name
        else:shutil.copy2(source,destination)
    resolver=lambda row:(BUNDLE_DIR/'input/inputs'/row.split/f'{row.key}.png') if BUNDLE_DIR else Path(row.crop_path)
    import contextlib, sys, platform
    from datetime import datetime, timezone
    class _Tee:
        def __init__(self,*streams):self.streams=streams
        def write(self,text):
            for stream in self.streams:stream.write(text)
            return len(text)
        def flush(self):
            for stream in self.streams:stream.flush()
    log_path=reproduction_dir/'reproduction_training_log.txt'
    with log_path.open('w',encoding='utf-8') as log_file, contextlib.redirect_stdout(_Tee(sys.stdout,log_file)):
        import torch
        print('Reproduction check; not used for model selection or reported results.')
        print('UTC start:',datetime.now(timezone.utc).isoformat())
        print('Python',platform.python_version(),'| torch',torch.__version__,
              '| device',torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu')
        reproduction_selection=train_project(reproduction_root,BASE,experiment=reproduction_name,
            input_resolver=resolver,legacy_archive=LEGACY_ARCHIVE)
        try:
            rows=[]
            for receipt_path in sorted((reproduction_dir/'models').glob('*.json')):
                formal_path=GPU/'models'/receipt_path.name
                if formal_path.is_file():
                    fresh=json.loads(receipt_path.read_text());formal=json.loads(formal_path.read_text())
                    rows.append({'head':receipt_path.stem,'formal_val_loss':formal.get('val_loss'),
                                 'reproduced_val_loss':fresh.get('val_loss'),'device':fresh.get('device')})
            print('Reproduced vs formal validation loss (consistency check only):')
            print(pd.DataFrame(rows).to_string(index=False) if rows else 'No comparable receipts found.')
        except Exception as error:
            print('Receipt comparison skipped:',type(error).__name__,error)
        print('UTC end:',datetime.now(timezone.utc).isoformat())
    print('Reproduction fits saved separately. Formal evaluation still uses the archived GPU models.')
    print('Training log:',log_path)
else:
    print('Reusing archived GPU models; no CNN retraining. Training histories and receipt hashes are shown below.')
training_receipts=[]
for path in sorted((GPU/'models').glob('*.json')):
    receipt=json.loads(path.read_text())
    assert receipt['device']=='cuda' and file_hash(path.with_suffix('.pt'))==receipt['sha256']
    training_receipts.append(receipt)
assert len(training_receipts)==12
display(pd.DataFrame(training_receipts)[['variant','lr','seed','val_loss','device']])


## 6. Fixed-model reproduction and lightweight controls
Test pairs were previously exposed in the Handiprint study, blood metadata diagnostics and the reference evaluation. This stage reproduces fixed predictions; it is not a new untouched-test claim. Validation losses must reproduce before the first test inference in this output directory. A completed evaluation lock reuses saved predictions on subsequent runs. An interrupted evaluation is retained for diagnosis.

Controls use training-only fitted encoders/scalers and fixed balanced logistic C=1. Categorical size fixes the earlier weak linear-size probe. Crop area is orientation-invariant. Area/brightness uses crop area, weighted sRGB code-value mean, its standard deviation, and exact-white fraction on the entire canvas. These are not background-only features. The combined probe is supplemental and was completed after primary test scores were known; no CNN, crop or threshold was changed.

The reference evaluation used saved GPU weights on CPU and reproduced selected validation losses within 5.3e-8. This run records its own inference device. Source suffix is metadata, not a CNN input. Probe predictiveness does not establish CNN reliance; poor probe accuracy would not establish absence of confounding.


In [ ]:
ANALYSIS_IMPLEMENTATION_SHA256='09933a174ba81748fa30b36cad6a0581a9df1c0ae630682e483ce434bc36cb19'
# Reproduce fixed-model inference; never select a CNN using test results.
from datetime import datetime, timezone
from types import SimpleNamespace
from importlib.metadata import version
import joblib
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import roc_auc_score

torch.set_num_threads(min(4,os.cpu_count() or 1))
torch.use_deterministic_algorithms(True)
WORK = FINAL_OUTPUT
WORK.mkdir(parents=True, exist_ok=True)
for folder in ['tables', 'figures', 'report']:
    (WORK / folder).mkdir(exist_ok=True)

def digest(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for block in iter(lambda: handle.read(1024*1024), b''):
            h.update(block)
    return h.hexdigest()

def write_json(data, path):
    path = Path(path)
    temporary = path.with_suffix(path.suffix + '.tmp')
    temporary.write_text(json.dumps(data, indent=2, default=str))
    temporary.replace(path)

selection = json.loads((GPU/'model_selection.json').read_text())
training_plan = json.loads((GPU/'experiment_plan.json').read_text())
training_environment = json.loads((GPU/'environment.json').read_text())
manifest = pd.read_csv(GPU/'dataset_manifest.csv', keep_default_na=False)
audit = pd.read_csv(BASE/'blood_manifest_audit.csv', keep_default_na=False).set_index('key')
assert len(manifest)==308 and manifest.key.is_unique
assert manifest.groupby('split').size().to_dict()=={'train':180,'val':64,'test':64}
assert manifest.groupby('pair_id').split.nunique().eq(1).all()
assert manifest.groupby('crop_sha256').split.nunique().eq(1).all()
assert digest(GPU/'dataset_manifest.csv')==training_plan['manifest_sha256']
assert digest(GPU/'external_crop_import.csv')==training_plan['heldout_annotations_sha256']
assert digest(BASE/'preprocessing_lock.json')==training_plan['crop_lock_sha256']
for name, expected in selection['model_sha256'].items():
    assert digest(GPU/'models'/name)==expected, name
receipts=[]
for receipt_path in sorted((GPU/'models').glob('*.json')):
    receipt=json.loads(receipt_path.read_text())
    assert receipt['device']=='cuda' and digest(receipt_path.with_suffix('.pt'))==receipt['sha256']
    receipts.append(receipt)
assert len(receipts)==12

# The uploaded training-only statistics are matched by processed-image hash.
statistics_path = TRAIN_STATISTICS
statistics=pd.read_csv(statistics_path)
training_rows=manifest[manifest.split.eq('train')]
assert len(statistics)==180 and statistics.key.is_unique
assert set(statistics.key)==set(training_rows.key)
assert statistics.set_index('key').crop_sha256.sort_index().equals(
    training_rows.set_index('key').crop_sha256.sort_index())
assert np.isfinite(statistics.select_dtypes('number')).all().all()
assert statistics.white_fraction.between(0,1).all()

def processed_path(row):
    if BUNDLE_DIR is None:
        return Path(row.crop_path)
    return BUNDLE_DIR/'input'/'inputs'/row.split/f'{row.key}.png'

feature_rows=statistics.to_dict('records')
for row in manifest[manifest.split.isin(['val','test'])].itertuples():
    path=processed_path(row)
    assert digest(path)==row.crop_sha256, row.key
    with Image.open(path) as image:
        assert image.mode=='RGB' and image.size==(224,224)
        rgb=np.asarray(image).astype(np.float64)
    luma=rgb@np.array([.2126,.7152,.0722]); means=rgb.mean(axis=(0,1))
    feature_rows.append({'key':row.key,'crop_sha256':row.crop_sha256,
        'mean_r':means[0],'mean_g':means[1],'mean_b':means[2],
        'luma_mean':luma.mean(),'luma_std':luma.std(),
        'white_fraction':np.all(rgb==255,axis=2).mean()})
features=manifest[['key','pair_id','split','y','brand','side','condition']].copy()
features['size_category']=features.key.map(audit['Size (US)']).astype(str)
features['collection_suffix']=features.key.str.replace(r'^\d{3}[lr]b[fp]t','',regex=True)
areas=[]
for row in manifest.itertuples():
    if row.split=='train':
        x0,y0,x1,y1=json.loads(row.bounds); value=(x1-x0)*(y1-y0)
    else:
        raw=audit.loc[row.key]
        value=float(row.width)*float(row.height)/(float(raw.width)*float(raw.height))
    areas.append(value)
features['crop_area']=areas
features=features.merge(pd.DataFrame(feature_rows),on='key',validate='one_to_one')
features.to_csv(WORK/'tables/diagnostic_features.csv',index=False)

analysis_plan={
    'formal_experiment':'blood_external_gpu_v1',
    'selected_models_sha256':digest(GPU/'model_selection.json'),
    'manifest_sha256':digest(GPU/'dataset_manifest.csv'),
    'training_statistics_sha256':digest(statistics_path),
    'threshold':.5,'bootstrap_pairs':2000,'bootstrap_seed':673,
    'ensemble':'mean of three seed probabilities',
    'probe_C':1.,'probe_class_weight':'balanced',
    'area_brightness_features':['crop_area','luma_mean','luma_std','white_fraction'],
    'test_exposure':'Prior Handiprint test reuse, blood metadata diagnostics and reference model test results already viewed; exploratory reproduction.',
    'area_brightness_status':'Supplemental diagnostic specified after primary model test results; not confirmatory.',
    'source_implementation_sha256':ANALYSIS_IMPLEMENTATION_SHA256,
}
plan_path=WORK/'analysis_plan.json'
if plan_path.exists():
    assert json.loads(plan_path.read_text())==analysis_plan, 'Preserve this analysis; changed settings require a new output directory.'
else:
    write_json(analysis_plan,plan_path)

# Fit only the four fixed lightweight controls on training rows.
tr=features[features.split.eq('train')]
probe_specs={
    'categorical_size':(['size_category'],True),
    'collection_suffix':(['collection_suffix'],True),
    'crop_area':(['crop_area'],False),
    'area_brightness':(analysis_plan['area_brightness_features'],False),
}
probes={}
for name,(columns,categorical) in probe_specs.items():
    encoder=OneHotEncoder(handle_unknown='ignore') if categorical else StandardScaler()
    probe=make_pipeline(encoder,LogisticRegression(C=1,class_weight='balanced',max_iter=3000,random_state=673))
    probe.fit(tr[columns],tr.y)
    probes[name]=(probe,columns)

DEVICE='cuda' if torch.cuda.is_available() else 'cpu'
network=FrozenCNN().to(DEVICE).eval()
backbone=Path(torch.hub.get_dir())/'checkpoints/resnet18-f37072fd.pth'
assert digest(backbone)==training_environment['backbone_weights_sha256']
hog_model=joblib.load(GPU/'models/hog.joblib')
variants={key:(value['heads'],key) for key,value in selection['variants'].items()}
variants['legacy_handiprint']=(selection['legacy_heads'],'gray')
reproduction=[]
seed_predictions=[]

def infer_split(split):
    rows=manifest[manifest.split.eq(split)].sort_values('key').copy()
    arrays=[]
    for row in rows.itertuples():
        path=processed_path(row)
        assert digest(path)==row.crop_sha256
        with Image.open(path) as image:
            assert image.mode=='RGB' and image.size==(224,224)
            arrays.append(np.asarray(image))
    images=np.stack(arrays)
    scores={'majority':np.full(len(rows),selection['majority_score']),
            'hog':hog_model.predict_proba(hog_matrix(images))[:,1]}
    for name,(heads,mode) in variants.items():
        estimates=[]
        for head in heads:
            network.head.load_state_dict(torch.load(GPU/'models'/head,map_location='cpu',weights_only=True))
            value=cnn_scores(network,images,mode);estimates.append(value)
            for key,y,score in zip(rows.key,rows.y,value):
                seed_predictions.append({'key':key,'split':split,'model':name,'head':head,'y':y,'score_nike':float(score)})
            if split=='val' and name!='legacy_handiprint':
                observed=float(log_loss(rows.y,np.clip(value,1e-7,1-1e-7),labels=[0,1]))
                expected=selection['fit_provenance'][head]['val_loss']
                reproduction.append({'head':head,'recorded':expected,'reproduced':observed,'absolute_difference':abs(expected-observed)})
        scores[name]=np.mean(estimates,axis=0)
    aligned=features.set_index('key').loc[rows.key]
    for name,(probe,columns) in probes.items():
        scores[name]=probe.predict_proba(aligned[columns])[:,1]
    tables=[]
    for name,value in scores.items():
        frame=rows[['key','pair_id','side','condition','brand','y','status']].copy()
        frame['model']=name;frame['score_nike']=value
        frame['prediction']=(value>.5).astype(int);tables.append(frame)
    return pd.concat(tables,ignore_index=True)

seal_path=WORK/'evaluation_lock.json'
if seal_path.exists():
    seal=json.loads(seal_path.read_text())
    assert seal['status']=='completed', 'Previous reproduction was interrupted. Preserve the record and diagnose before rerunning.'
    assert seal['analysis_plan_sha256']==digest(plan_path)
    for split in ['val','test']:
        assert digest(WORK/f'tables/{split}_predictions.csv')==seal[f'{split}_predictions_sha256']
    validation=pd.read_csv(WORK/'tables/val_predictions.csv')
    test=pd.read_csv(WORK/'tables/test_predictions.csv')
    print('Verified and reused saved predictions. No CNN training or new test inference.')
else:
    validation=infer_split('val')
    assert max(row['absolute_difference'] for row in reproduction)<1e-4, 'Stop before test: validation does not reproduce.'
    pd.DataFrame(reproduction).to_csv(WORK/'tables/validation_reproduction.csv',index=False)
    validation.to_csv(WORK/'tables/val_predictions.csv',index=False)
    seal={'status':'started','started_at_utc':datetime.now(timezone.utc).isoformat(),
          'analysis_plan_sha256':digest(plan_path),'inference_device':DEVICE}
    write_json(seal,seal_path)
    test=infer_split('test')
    test.to_csv(WORK/'tables/test_predictions.csv',index=False)
    pd.DataFrame(seed_predictions).to_csv(WORK/'tables/seed_predictions.csv',index=False)
    seal.update(status='completed',completed_at_utc=datetime.now(timezone.utc).isoformat(),
        val_predictions_sha256=digest(WORK/'tables/val_predictions.csv'),
        test_predictions_sha256=digest(WORK/'tables/test_predictions.csv'))
    write_json(seal,seal_path)
    print('Fixed-model predictions reproduced and saved. No models selected on test.')
write_json({'inference_device':DEVICE,'packages':{name:version(name) for name in
    ['torch','torchvision','numpy','pandas','pillow','scikit-learn','scikit-image','joblib']},
    'training_environment':training_environment},WORK/'inference_environment.json')


## 7. Shoe-pair uncertainty and sensitivity
Bootstrap resamples whole pairs within class, preserving sides and Full/Partial observations. The same 2000 draws (seed 673) support paired model differences and paired condition differences. Primary BA counts unscorable cases as errors, with conditional metrics and coverage reported separately; no prepared inputs failed in this study.

Intervals condition on the observed split and fitted models, omit training/split/session variability and are exploratory without multiplicity correction. A [1,1] interval from all-correct observed pairs is not a population guarantee. Zero observed CNN differences do not establish equivalence. Sensitivity excludes the seven metadata-inferred pairs from the same predictions, not from retraining; two occur in test and two in validation.


In [ ]:
def conditional_loss(y, score):
    return float(log_loss(y,np.clip(score,1e-7,1-1e-7),labels=[0,1]))

def pair_bootstrap_counts(frame, repetitions, seed):
    pairs=frame[['pair_id','y']].drop_duplicates().sort_values('pair_id')
    assert pairs.groupby('pair_id').y.nunique().eq(1).all()
    rng=np.random.default_rng(seed); counts=np.zeros((repetitions,len(pairs)),dtype=np.float32)
    for label in [0,1]:
        indexes=np.flatnonzero(pairs.y.to_numpy()==label)
        assert len(indexes)>0
        counts[:,indexes]=rng.multinomial(len(indexes),np.full(len(indexes),1/len(indexes)),size=repetitions)
    return pairs,counts


def summarize_predictions(frame, repetitions=2000, seed=673):
    from sklearn.metrics import roc_auc_score
    base=frame[['key','pair_id','y','condition']].drop_duplicates().sort_values('key')
    assert base.key.is_unique
    pairs,counts=pair_bootstrap_counts(base,repetitions,seed)
    pair_index={pair:index for index,pair in enumerate(pairs.pair_id)}
    summaries=[]; bootstrap={}
    for model in sorted(frame.model.unique()):
        rows=frame[frame.model.eq(model)].set_index('key').loc[base.key].reset_index()
        for condition in ['All','Full','Partial']:
            subset=rows if condition=='All' else rows[rows.condition.eq(condition)]
            y=subset.y.to_numpy(); score=subset.score_nike.to_numpy(dtype=float)
            scorable=np.isfinite(score); predicted=np.where(scorable,(score>.5).astype(int),-1)
            weights=counts[:,[pair_index[pair] for pair in subset.pair_id]]
            boot=np.zeros(repetitions); recalls=[]
            for label in [0,1]:
                mask=y==label; assert mask.any()
                correct=predicted[mask]==label
                recalls.append(float(np.mean(correct)))
                boot += .5*(weights[:,mask]*correct).sum(axis=1)/weights[:,mask].sum(axis=1)
            ba=float(np.mean(recalls)); lower,upper=np.percentile(boot,[2.5,97.5])
            bootstrap[(model,condition)]=boot
            conditional_ba=np.nan
            if set(y[scorable])=={0,1}:
                conditional_ba=.5*sum(np.mean(predicted[scorable & (y==label)]==label) for label in [0,1])
            summaries.append({'model':model,'condition':condition,'n_images':len(subset),'n_pairs':subset.pair_id.nunique(),
                              'coverage':float(scorable.mean()),'balanced_accuracy':ba,'ci_low':float(lower),'ci_high':float(upper),
                              'recall_adidas':recalls[0],'recall_nike':recalls[1],'conditional_balanced_accuracy':conditional_ba,
                              'conditional_logloss':conditional_loss(y[scorable],score[scorable]) if scorable.any() else np.nan,
                              'conditional_auc':float(roc_auc_score(y[scorable],score[scorable])) if set(y[scorable])=={0,1} else np.nan})
    summary=pd.DataFrame(summaries); differences=[]
    comparisons=[('rgb_occlusion','rgb'),('gray','rgb'),('rgb','hog')]
    if 'legacy_handiprint' in set(frame.model): comparisons.append(('rgb','legacy_handiprint'))
    if 'area_brightness' in set(frame.model): comparisons.append(('rgb','area_brightness'))
    for left,right in comparisons:
        if left not in set(frame.model) or right not in set(frame.model): continue
        for condition in ['All','Full','Partial']:
            draws=bootstrap[(left,condition)]-bootstrap[(right,condition)]
            point=summary.set_index(['model','condition']).loc[(left,condition),'balanced_accuracy']-summary.set_index(['model','condition']).loc[(right,condition),'balanced_accuracy']
            lo,hi=np.percentile(draws,[2.5,97.5])
            differences.append({'comparison':left+' minus '+right,'condition':condition,'difference':float(point),'ci_low':float(lo),'ci_high':float(hi)})
    for model in set(frame.model):
        draws=bootstrap[(model,'Partial')]-bootstrap[(model,'Full')]
        indexed=summary.set_index(['model','condition']);point=indexed.loc[(model,'Partial'),'balanced_accuracy']-indexed.loc[(model,'Full'),'balanced_accuracy']
        lo,hi=np.percentile(draws,[2.5,97.5])
        differences.append({'comparison':model+' Partial minus Full','condition':'paired conditions','difference':float(point),'ci_low':float(lo),'ci_high':float(hi)})
    return summary,pd.DataFrame(differences)



all_metrics=[];all_differences=[]
for split,frame in [('val',validation),('test',test)]:
    for cohort in ['all','exclude_metadata_inferred']:
        selected=frame if cohort=='all' else frame[~frame.pair_id.isin([9,70,76,79,114,120,158])]
        metrics,differences=summarize_predictions(selected,2000,673)
        metrics['split']=split;metrics['cohort']=cohort
        differences['split']=split;differences['cohort']=cohort
        all_metrics.append(metrics);all_differences.append(differences)
metrics=pd.concat(all_metrics,ignore_index=True)
differences=pd.concat(all_differences,ignore_index=True)
metrics.to_csv(WORK/'tables/metrics.csv',index=False)
differences.to_csv(WORK/'tables/paired_differences.csv',index=False)
print('Test balanced accuracy; 64 images / 16 pairs:')
display(metrics.query("split=='test' and cohort=='all' and condition=='All'")[[
    'model','balanced_accuracy','ci_low','ci_high','recall_adidas','recall_nike','conditional_logloss']].round(4))
print('Full/partial and sensitivity results:')
display(metrics.query("split=='test'")[[
    'cohort','model','condition','n_images','n_pairs','balanced_accuracy']].round(4))
print('Paired differences (percentage-point interpretation; exploratory):')
display(differences.query("split=='test' and cohort=='all'").round(4))


## 8. Interpretation, model-use activity, figures and report
The additional activity compares mean Nike scores on full versus partial impressions within each shoe pair, averaging sides. It examines transfer failures and score shifts rather than claiming new independent-case validation. Scores are uncalibrated closed-set outputs, not forensic likelihoods or wearer identification.

All three blood CNNs have 100% observed BA in the reference evaluation, so no accuracy benefit of grayscale or occlusion augmentation is established. Collection suffix, shoe size and crop geometry/brightness are predictive; CNN success cannot be assigned specifically to tread texture. The dataset has one style per brand. A future confirmatory design needs new independent shoes and sessions balanced across classes, rather than more test-driven tuning.

The report below combines archived Handiprint findings with the actual blood extension. The report includes data examples, training curves, archived Handiprint evidence, current model results and diagnostic controls. It records the assistance used without asserting instructor approval.


In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch
from IPython.display import display, Markdown, Image as DisplayImage
import io, zipfile, importlib.util, subprocess, sys

labels={'majority':'Majority','hog':'HOG + logistic','legacy_handiprint':'Handiprint transfer',
    'rgb':'RGB CNN','gray':'Grayscale CNN','rgb_occlusion':'RGB + occlusion',
    'categorical_size':'Categorical size','collection_suffix':'Collection suffix',
    'crop_area':'Crop area','area_brightness':'Area + brightness'}
order=['majority','legacy_handiprint','hog','rgb','gray','rgb_occlusion',
       'categorical_size','crop_area','area_brightness','collection_suffix']
main=metrics.query("split=='test' and cohort=='all' and condition=='All'").set_index('model').loc[order]
plt.rcParams.update({'font.size':10,'axes.spines.top':False,'axes.spines.right':False})
fig,ax=plt.subplots(figsize=(10,5.5));y=np.arange(len(main));values=main.balanced_accuracy.to_numpy()
ax.barh(y,values,color=['#64748b']*3+['#2563eb']*3+['#b45309']*4)
ax.errorbar(values,y,xerr=np.array([values-main.ci_low.to_numpy(),main.ci_high.to_numpy()-values]),fmt='none',ecolor='black',capsize=3)
ax.set_yticks(y,[labels[x] for x in main.index]);ax.invert_yaxis()
ax.set(xlim=(0,1.24),xlabel='Balanced accuracy (64 images; 16 pairs)',title='Blood-on-tile: fixed models and diagnostic controls')
for i,v in enumerate(values):ax.text(1.14,i,f'{v:.1%}',va='center',fontsize=9)
fig.text(.5,.01,'95% pair-bootstrap intervals; a perfect-sample interval is not a population guarantee.',ha='center',fontsize=8)
fig.tight_layout(rect=(0,.03,1,1));fig.savefig(WORK/'figures/test_balanced_accuracy.png',dpi=170);plt.show()

fig,axes=plt.subplots(1,3,figsize=(11,3.3),sharey=True)
for ax,name in zip(axes,['rgb','gray','rgb_occlusion']):
    for head in selection['variants'][name]['heads']:
        frame=pd.read_csv(GPU/'models'/Path(head).with_suffix('.csv'))
        ax.plot(frame.epoch,frame.val_logloss,label='seed '+head.split('seed')[-1].split('.')[0])
    ax.set(title=labels[name],xlabel='Epoch',yscale='log');ax.legend(fontsize=8)
axes[0].set_ylabel('Validation log-loss (model selection)')
fig.tight_layout();fig.savefig(WORK/'figures/validation_curves.png',dpi=170);plt.show()

activity=test[test.model.isin(['legacy_handiprint','rgb'])].groupby(
    ['model','pair_id','brand','condition']).score_nike.mean().unstack('condition').reset_index()
activity['partial_minus_full']=activity.Partial-activity.Full
activity.to_csv(WORK/'tables/pair_score_activity.csv',index=False)
fig,axes=plt.subplots(1,2,figsize=(8,3.5),sharex=True,sharey=True)
for ax,name in zip(axes,['legacy_handiprint','rgb']):
    for brand,color in [('Adidas','#f97316'),('Nike','#2563eb')]:
        frame=activity[activity.model.eq(name)&activity.brand.eq(brand)]
        ax.scatter(frame.Full,frame.Partial,label=brand,color=color,s=36,alpha=.85)
    ax.plot([0,1],[0,1],':',color='gray');ax.axhline(.5,color='gray',lw=.6);ax.axvline(.5,color='gray',lw=.6)
    ax.set(title=labels[name],xlabel='Mean Nike score: full');ax.legend(fontsize=8)
axes[0].set_ylabel('Mean Nike score: partial')
fig.suptitle('Model-use activity: paired score changes, sides averaged')
fig.tight_layout();fig.savefig(WORK/'figures/pair_score_activity.png',dpi=170);plt.show()

# Show observed errors rather than inventing CNN failures.
errors=test[test.prediction.ne(test.y)].copy()
errors.to_csv(WORK/'tables/misclassifications.csv',index=False)
confusion=[]
for model,group in test.groupby('model'):
    for actual in [0,1]:
        for predicted in [0,1]:
            confusion.append({'model':model,'actual':actual,'predicted':predicted,
                'count':int(((group.y==actual)&(group.prediction==predicted)).sum())})
pd.DataFrame(confusion).to_csv(WORK/'tables/confusion_counts.csv',index=False)
fig,axes=plt.subplots(1,3,figsize=(9,2.8))
for ax,model in zip(axes,['legacy_handiprint','hog','rgb']):
    group=test[test.model.eq(model)]
    matrix=np.array([[int(((group.y==i)&(group.prediction==j)).sum()) for j in [0,1]] for i in [0,1]])
    ax.imshow(matrix,cmap='Blues',vmin=0,vmax=32)
    for i in [0,1]:
        for j in [0,1]:ax.text(j,i,str(matrix[i,j]),ha='center',va='center',color='white' if matrix[i,j]>16 else 'black')
    ax.set_xticks([0,1],['Adidas','Nike']);ax.set_yticks([0,1],['Adidas','Nike']);ax.set(title=labels[model],xlabel='Predicted')
axes[0].set_ylabel('Actual');fig.tight_layout();fig.savefig(WORK/'figures/confusion_matrices.png',dpi=170);plt.show()

# A small case review supports the additional model-use activity.
example_rows=errors[errors.model.eq('legacy_handiprint')].sort_values('score_nike').head(4)
if len(example_rows):
    fig,axes=plt.subplots(1,len(example_rows),figsize=(10,3))
    for ax,row in zip(np.atleast_1d(axes),example_rows.itertuples()):
        original=manifest.set_index('key').loc[row.key]
        from types import SimpleNamespace
        path=processed_path(SimpleNamespace(crop_path=original.crop_path,split='test',key=row.key))
        with Image.open(path) as image:ax.imshow(image)
        ax.set_title(f'Pair {row.pair_id}: {row.condition}\nTrue {row.brand}; Nike score {row.score_nike:.3f}',fontsize=8);ax.axis('off')
    fig.suptitle('Observed Handiprint-transfer errors; not independent new cases')
    fig.tight_layout();fig.savefig(WORK/'figures/transfer_error_examples.png',dpi=170);plt.show()
print('Blood CNN errors observed:',int(errors.model.isin(['rgb','gray','rgb_occlusion']).sum()))

# Architecture diagram: only the binary head is trained.
fig,ax=plt.subplots(figsize=(10,1.8));ax.set_xlim(0,10);ax.set_ylim(0,2);ax.axis('off')
for x,text,color in [(0,'RGB224 canvas\naspect preserved','#dbeafe'),(2.6,'ImageNet normalization\nchannel-wise','#e2e8f0'),(5.2,'Frozen ResNet18\n512 features','#dbeafe'),(7.8,'Trainable linear head\n513 parameters + sigmoid','#fed7aa')]:
    ax.add_patch(FancyBboxPatch((x,.5),2.1,1,boxstyle='round,pad=.08',facecolor=color,edgecolor='#475569'))
    ax.text(x+1.05,1,text,ha='center',va='center',fontsize=8)
    if x<7.8:ax.annotate('',xy=(x+2.5,1),xytext=(x+2.15,1),arrowprops={'arrowstyle':'->'})
fig.tight_layout();fig.savefig(WORK/'figures/architecture.png',dpi=170);plt.show()

pct=lambda x:f'{100*float(x):.2f}%'
ci=lambda row:f'[{pct(row.ci_low)}, {pct(row.ci_high)}]'
fullpartial=metrics.query("split=='test' and cohort=='all'").set_index(['model','condition'])
probe_rows=[['Control','Validation BA','Test BA','Test 95% pair CI']]
for model in ['categorical_size','crop_area','area_brightness','collection_suffix']:
    v=metrics.query("split=='val' and cohort=='all' and condition=='All' and model==@model").iloc[0]
    t=main.loc[model];probe_rows.append([labels[model],pct(v.balanced_accuracy),pct(t.balanced_accuracy),ci(t)])
model_rows=[['Model','Test BA','Adidas recall','Nike recall']]
for model in ['legacy_handiprint','hog','rgb','gray','rgb_occlusion']:
    t=main.loc[model];model_rows.append([labels[model],pct(t.balanced_accuracy),pct(t.recall_adidas),pct(t.recall_nike)])
condition_rows=[['Model','Full BA','Partial BA']]
for model in ['legacy_handiprint','hog','rgb','gray','rgb_occlusion']:
    condition_rows.append([labels[model],pct(fullpartial.loc[(model,'Full'),'balanced_accuracy']),pct(fullpartial.loc[(model,'Partial'),'balanced_accuracy'])])

# Six report pages plus a references page. All statements separate archived
# Handiprint findings from the newly reproduced blood-extension results.
pages=[
 {'title':'Footwear patterns under occlusion and cross-medium transfer', 'blocks':[
  ('p','Dora Zhuang | MGSC 673 | Exploratory individual project'),
  ('h','Problem and research questions'),
  ('p','Incomplete footwear impressions lose tread information and acquire substrate and collection artefacts. This project investigates two specific patterns: Adidas Seeley and Nike Zoom Winflo 4. It asks how simulated information loss affects classification, how an existing Handiprint classifier transfers to related-pair blood impressions, and whether grayscale or occlusion augmentation changes performance on real full/partial blood inputs.'),
  ('h','Data and preparation'),
  ('p','The ShoeCase Handiprint metadata contain 156 records from 78 pairs. Excluding pair 81, which lacks one required Handiprint source image, leaves 154 images / 77 pairs. The completed original study uses 90 training, 32 validation and 32 test images. The tile/synthetic-blood subset contains 312 JPGs / 78 candidate pairs; retaining the same exclusion leaves 308 images / 77 pairs. The blood images are repeated media/conditions from related shoes, not 312 independent shoes.'),
  ('table',[['Blood split','Images','Pairs','Adidas / Nike pairs'],['Train','180','45','23 / 22'],['Validation','64','16','8 / 8'],['Test','64','16','8 / 8']]),
  ('p','Pair membership is preserved across media, sides and conditions. Seven links (9, 70, 76, 79, 114, 120, 158) are metadata-inferred rather than independently verified. Pair 157 retains its metadata-declared Nike label, with four source-name/date discrepancies disclosed.'),
  ('p','Red localization was investigated but not adopted as a reliable automatic method. Training blood crops use 180 approved manual RGB ROIs. The held-out images were cropped on desktop under the same stated content-preservation rules. Native orientation is retained without perspective warping; aspect ratio is preserved on a white 224-pixel canvas. Different cropping interfaces and retained substrate remain limitations. Two historical review packages jointly cover all 180 training keys, and latest records match the locked annotations.') ]},
 {'title':'Model implementation and experimental discipline','blocks':[
  ('image','architecture.png',6.5,1.2),
  ('p','The blood CNNs use ImageNet ResNet18 with all backbone parameters frozen and a newly trained 512-to-1 linear head. Only 513 parameters are optimized. RGB224 arrays are converted NHWC to NCHW and normalized by the ImageNet channel means and standard deviations. Grayscale uses the same crop converted to PIL luminance and repeated across three channels. Batch-normalization layers stay in evaluation mode.'),
  ('table',[['Setting','Blood experiment'],['Optimizer / loss','Adam; class-weighted binary cross-entropy with logits'],['Learning rates','RGB: 0.001, 0.01; validation selects 0.01'],['Controlled variants','Gray and RGB-occlusion reuse RGB-selected rate'],['Seeds / ensemble','0, 1, 2; mean of sigmoid probabilities'],['Training budget','Batch 16; up to 40 epochs; patience 7'],['Checkpoint','Validation log-loss improvement greater than 0.0001'],['Regularization','Weight decay 0.0001; modest affine/colour augmentation'],['Occlusion','Probability 0.5; area 0.1-0.7 of the padded canvas'],['Decision threshold','0.5, fixed before CNN test evaluation']]),
  ('p','HOG uses grayscale 128-pixel inputs, training-fitted standardization and a balanced logistic classifier; validation selects C=0.1. The fixed transfer comparator reconstructs the original saved Handiprint heads with the same ImageNet backbone. It is the original Handiprint model, not the severe-occlusion-augmented checkpoint, which was not supplied in the saved archive.'),
  ('p','All twelve blood CNN candidate fits in blood_external_gpu_v1 have CUDA receipts. The earlier CPU/mixed directory is retained but is not selected for reporting. GPU training loss histories are preserved. Reference inference on CPU reproduced selected-head validation losses within 5.3e-8 before test prediction. The notebook records its reproduction device and package versions separately. No CNN is chosen by test score.') ]},
 {'title':'Results: archived study and blood transfer','blocks':[
  ('h','Completed Handiprint phase (archived results)'),
  ('p','The original main study reported 100% unmodified test balanced accuracy for HOG and the frozen-backbone CNN on 32 images / 16 pairs. A previously completed exploratory severe-mask extension reported 53.75% BA for the original CNN and 95.00% for occlusion-augmented training at 80% crop-area masking: +41.25 percentage points, paired 95% shoe-pair bootstrap interval [35.63, 46.88]. These values are retained from the supplied original report and notebook outputs, not recomputed from unavailable Handiprint TIFFs or augmented checkpoints.'),
  ('h','Blood-on-tile phase (actual saved-model evaluation)'),
  ('table',model_rows),
  ('p','All three blood-trained ensembles correctly classify all 64 observed test images. HOG makes 5 errors; the original Handiprint transfer model makes 18, all on Nike. RGB minus HOG is +7.81 percentage points [3.13, 14.06]; RGB minus Handiprint transfer is +28.13 points [15.63, 40.63], using 2000 shared class-stratified whole-pair bootstrap draws.'),
  ('p','The three blood CNNs have zero observed BA differences. This is a ceiling effect, not evidence of equivalence or of an augmentation benefit. Transfer combines medium, substrate, acquisition and preprocessing changes; the improvement is not an isolated causal effect of medium adaptation. Historical 80% Handiprint crop-area masking and current canvas-area augmentation are different interventions and should not be compared as identical severity.'),
  ('p','Removing metadata-inferred test pairs 70 and 120 leaves 56 images / 14 pairs. All three blood CNNs remain at 100%; HOG is 91.07% and Handiprint transfer 73.21%. These are filters of the same frozen predictions, not retrained models.') ]},
 {'title':'Diagnostic controls and what high accuracy means','blocks':[
  ('table',probe_rows),
  ('p','Categorical size and source-suffix encoders are fitted on training data only. Crop area uses normalized training ROI area and desktop crop pixel area divided by original pixel area, avoiding the EXIF width/height swap problem. Every control is a fixed C=1 balanced logistic regression; numerical features use a training-fitted StandardScaler. There is no probe hyperparameter search.'),
  ('p','The area/brightness control uses four features: crop area, mean weighted sRGB code value, its standard deviation, and exact-white pixel fraction. Statistics describe the entire RGB224 canvas including padding, not background-only illumination. All 180 uploaded training summaries match their processed-image hashes. The combined control was completed after primary test results were known and is explicitly supplemental and exploratory; no CNN or crop was changed in response.'),
  ('p','Source suffix predicts the test labels perfectly, and size predicts them at 87.5%. Area and brightness also carry class information. These controls demonstrate that detailed local tread texture is not the only potentially useful signal. They do not establish which features a CNN used. Shoe size/shape can be legitimate pattern attributes, and white padding can reflect the cropping protocol; neither should automatically be called an unrelated background cue.'),
  ('p','Metadata suffix is not supplied to the image CNN. Its perfect prediction nevertheless reveals a brand/collection association that the present split cannot disentangle. Lower probe performance than CNN performance does not prove the CNN avoids confounding. A reliable background-only deep control would need validated foreground exclusion and was not added merely to increase experiment count.') ]},
 {'title':'Using the trained model: paired score behaviour','blocks':[
  ('image','pair_score_activity.png',6.3,2.5),
  ('table',condition_rows),
  ('p','The additional model-use activity averages the two sides within each pair and plots mean Nike score on full versus partial impressions. This examines behaviour beyond a single aggregate score. In the reference evaluation, Handiprint-transfer Nike scores average 0.486 on full and 0.400 on partial inputs, close to or below the fixed threshold. RGB scores average 0.982 and 0.972. Adidas scores remain low. These are descriptive score shifts, not calibrated probabilities of brand identity or independent new-case validation.'),
  ('p','The notebook displays actual Handiprint-transfer errors and confusion matrices; it does not fabricate CNN failures when none occur. The paired Full/Partial BA difference for the transfer model is -6.25 points with an interval spanning zero, so the small sample does not establish a reliable partial-condition penalty. HOG also does not consistently worsen on partial inputs.'),
  ('p','Together, the archived severe-mask finding and current real-impression ceiling show why simulated robustness and real collection conditions need separate evaluation. The supported new result is successful classification within this related-pair cohort, with unresolved cues and no observed accuracy advantage of colour or canvas occlusion.') ]},
 {'title':'Limitations, ethical use and reproducibility','blocks':[
  ('h','Limits on interpretation'),
  ('p','The cohort contains two styles and only 16 related test pairs. Prior test exposure and collection/class association make all results exploratory. Validation supports stopping and selection, not an unbiased generalization estimate. Pair-bootstrap intervals condition on fixed models and this split; they omit training, split and session variability. A [100%,100%] interval on all-correct observed pairs is not a population guarantee.'),
  ('p','Cropping retains substrate information and differs operationally between train and held-out data. Exact file hashes do not replace near-duplicate or physical-identity review. Seven mappings and pair 157 remain documented provenance limitations. Confirmation needs new independent shoes/styles, size balance and acquisition sessions crossed with class; increasing model size or adding test-driven preprocessing would not resolve these design limits.'),
  ('h','Ethical considerations'),
  ('p','This is research on two footwear patterns, not wearer identification or forensic source attribution. A false confident prediction could misdirect an investigation. Closed-set scores are uncalibrated, cannot reject all unseen shoes and must not be treated as evidential likelihoods. Deployment would require representative data, unknown-class rejection, calibration, examiner review and measured errors across media and collection conditions.'),
  ('h','Reproducibility and AI assistance'),
  ('p','The single notebook contains data checks, frozen preprocessing, training code, saved GPU receipts, fixed-model evaluation, grouped uncertainty, controls, figures and the score activity. Default execution reuses saved models; it does not repeat manual review, downloads or test-based selection. Data access is through ShoeCase and the documented local Drive layout. The original split, locks and old runs remain unchanged. Restart/run all in the final Colab environment and save visible outputs before submission.'),
  ('p','AI assistance: Claude and OpenAI Codex assisted with code, provenance checks, experiment implementation, analysis and report drafting. Dora performed the crop reviews and ran the recorded training. The use of these tools is disclosed as part of the project record.') ]},
 {'title':'References and data access','blocks':[
  ('p','Tibben, A., Carriquiry, A., & McGuire, M. (2023). ShoeCase: A data set of mock crime scene footwear impressions. Iowa State University. https://doi.org/10.25380/iastate.22814438.v1'),
  ('p','He, K., Zhang, X., Ren, S., & Sun, J. (2016). Deep residual learning for image recognition. CVPR. https://www.cv-foundation.org/openaccess/content_cvpr_2016/html/He_Deep_Residual_Learning_CVPR_2016_paper.html'),
  ('p','Dalal, N., & Triggs, B. (2005). Histograms of oriented gradients for human detection. CVPR. https://doi.org/10.1109/CVPR.2005.177'),
  ('p','Zhong, Z., Zheng, L., Kang, G., & Yang, Y. (2017). Random erasing data augmentation. https://arxiv.org/abs/1708.04896'),
  ('p','Data access: the project package contains all 308 prepared RGB224 inputs and uses relative paths without personal Drive credentials. Exact source manifests, training summaries, GPU heads, review histories and the legacy results archive are included. The source DOI identifies the full dataset, which need not be downloaded for this reproduction. The README documents preprocessing transformations, dependency versions and the optional separate retraining procedure.') ]},
]


from types import SimpleNamespace

# Post-selection visual comparison for the reader; no crops or models are tuned.
examples=[]
for split in ['train','val']:
    for brand in ['Adidas','Nike']:
        for condition in ['Full','Partial']:
            selected=manifest[manifest.split.eq(split)&manifest.brand.eq(brand)&manifest.condition.eq(condition)].sort_values(['pair_id','side']).iloc[0]
            examples.append(selected)
fig,axes=plt.subplots(2,4,figsize=(10,5.5))
for ax,row in zip(axes.ravel(),examples):
    path=processed_path(SimpleNamespace(crop_path=row.crop_path,split=row.split,key=row.key))
    with Image.open(path) as image:ax.imshow(image)
    ax.set_title(f'{row.split}: {row.brand}, pair {row.pair_id}\n{row.condition}',fontsize=8);ax.axis('off')
fig.suptitle('Prepared inputs by class and condition: train (top), validation (bottom)')
fig.tight_layout();fig.savefig(WORK/'figures/train_validation_examples.png',dpi=170);plt.show()

fig,ax=plt.subplots(figsize=(7.5,3.0))
ax.plot(archived_severe.mask_percent,archived_severe.original_ba_percent,'o-',label='Original Handiprint CNN',color='#64748b')
ax.plot(archived_severe.mask_percent,archived_severe.augmented_ba_percent,'s-',label='Occlusion-augmented Handiprint CNN',color='#2563eb')
ax.axhline(50,color='gray',ls=':',lw=.8)
ax.set(xlabel='Target rectangular area (% of resized Handiprint crop)',ylabel='Balanced accuracy (%)',ylim=(40,105),title='Archived severe-mask experiment (rounded report values)')
ax.legend(fontsize=8);fig.tight_layout();fig.savefig(WORK/'figures/archived_handiprint_severe.png',dpi=170);plt.show()

experiment_log=pd.DataFrame([
    ['Handiprint: modest masking','CNN remained at 100%','Ceiling limited robustness conclusions; investigated severe masks'],
    ['Handiprint: 80% crop-area masks','Original 53.75%, augmented 95.00%','Archived exploratory benefit; do not equate with real partial contact'],
    ['Blood preprocessing pilot','Red localization unreliable','Used reviewed RGB ROIs; preserved disconnected detail'],
    ['Handiprint head on blood','Test BA 71.88%, Nike recall 43.75%','Investigated blood-adapted heads with same pair split'],
    ['Blood RGB / gray / occlusion','All test BA 100%','No observed BA advantage; no test-driven architecture search'],
    ['Size / suffix / crop controls','Size 87.5%, suffix 100%, area+brightness 79.69%','Restricted conclusions to this collection-confounded two-style cohort']],
    columns=['Experiment or issue','Observation','Decision and interpretation'])
experiment_log.to_csv(WORK/'tables/experiment_decision_log.csv',index=False)
display(experiment_log)


# Continuous report layout, rather than forced sparse pages.
pages[0]['blocks'].append(('image','train_validation_examples.png',6.6,3.63))
pages[0]['blocks'].append(('p','Prepared train and validation examples are displayed after model selection for documentation, without revising crops. They show substrate, impression strength and shape differences that motivate the diagnostic controls.'))
pages[1]['blocks'].append(('image','validation_curves.png',6.6,1.98))
pages[1]['blocks'].append(('p','Selected-rate validation curves for all three seeds; curves are model-selection evidence, not independent performance estimates.'))
pages[2]['blocks'].insert(2,('image','archived_handiprint_severe.png',6.6,2.64))
pages[2]['blocks'].append(('image','confusion_matrices.png',6.6,2.05))
pages[3]['blocks'].insert(0,('image','test_balanced_accuracy.png',6.6,3.63))
pages[4]['blocks'].append(('h','Experiment and decision record'))
pages[4]['blocks'].append(('table',[experiment_log.columns.tolist(),*experiment_log.values.tolist()]))
pages[5]['blocks']=[block for block in pages[5]['blocks'] if block[0]!='p' or not block[1].startswith('The single notebook contains')]
pages[5]['blocks'].append(('p','The extracted package includes all 308 processed RGB224 inputs and automatically resolves relative paths without personal Drive credentials. Hash-locked source manifests are preserved unchanged. Optional fresh training uses a separate output directory; default execution uses the archived CUDA fits and records the inference device. The submission ZIP is built from the saved, completed notebook after Run all, avoiding a snapshot of a running export cell.'))

# Avoid repeating the same model numbers in both a table and two figures.
pages[2]['blocks']=[block for block in pages[2]['blocks'] if not (block[0]=='table' and block[1] is model_rows)]
pages[2]['blocks'].insert(4,('p','HOG recalls are 90.63% for Adidas and 93.75% for Nike. The transferred model recalls are 100% and 43.75%; confusion counts are shown below. All three blood CNN recalls are 100% on this observed test set.'))

# Write a readable Markdown report.
report_lines=[]
for page in pages:
    report_lines.append('# '+page['title'])
    for block in page['blocks']:
        if block[0]=='p':report_lines.append(block[1])
        elif block[0]=='h':report_lines.append('## '+block[1])
        elif block[0]=='table':
            rows=block[1];report_lines.append('| '+' | '.join(rows[0])+' |\n| '+' | '.join(['---']*len(rows[0]))+' |\n'+'\n'.join('| '+' | '.join(row)+' |' for row in rows[1:]))
        elif block[0]=='image':report_lines.append(f'![{block[1]}](../figures/{block[1]})')
    report_lines.append('')
(WORK/'report/MGSC673_Footwear_Report.md').write_text('\n\n'.join(report_lines))

missing=[package for module,package in [('docx','python-docx'),('reportlab','reportlab')]
         if importlib.util.find_spec(module) is None]
if missing:subprocess.check_call([sys.executable,'-m','pip','install','-q',*missing])
from docx import Document
from docx.shared import Inches, Pt
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle, Image as PDFImage, PageBreak
from reportlab.lib import colors
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.enums import TA_LEFT
from xml.sax.saxutils import escape

doc=Document();normal=doc.styles['Normal'];normal.font.name='Calibri';normal.font.size=Pt(10)
normal.paragraph_format.space_after=Pt(5)
for section in doc.sections:
    section.top_margin=Inches(.65);section.bottom_margin=Inches(.65)
# Embed the Matplotlib-bundled font for consistent PDF rendering.
from reportlab.pdfbase import pdfmetrics
from reportlab.pdfbase.ttfonts import TTFont
import matplotlib
font_folder=Path(matplotlib.get_data_path())/'fonts/ttf'
pdfmetrics.registerFont(TTFont('MGSCBody',str(font_folder/'DejaVuSans.ttf')))
pdfmetrics.registerFont(TTFont('MGSCBold',str(font_folder/'DejaVuSans-Bold.ttf')))
styles=getSampleStyleSheet();styles.add(ParagraphStyle(name='BodyCompact',fontName='MGSCBody',fontSize=9.2,leading=12,spaceAfter=6))
styles.add(ParagraphStyle(name='TableCompact',fontName='MGSCBody',fontSize=8.7,leading=10.5,spaceAfter=0))
styles.add(ParagraphStyle(name='PageTitle',fontName='MGSCBold',fontSize=15,leading=18,spaceAfter=12))
styles.add(ParagraphStyle(name='SubheadCompact',fontName='MGSCBold',fontSize=10.5,leading=13,spaceAfter=6,spaceBefore=5))
story=[]
for i,page in enumerate(pages):
    # Sections flow continuously; only references start a new page.
    if i and page['title']=='References and data access':
        doc.add_page_break();story.append(PageBreak())
    doc.add_heading(page['title'],0 if i==0 else 1)
    story.append(Paragraph(escape(page['title']),styles['PageTitle']))
    for block in page['blocks']:
        kind=block[0]
        if kind=='p':
            doc.add_paragraph(block[1]);story.append(Paragraph(escape(block[1]),styles['BodyCompact']))
        elif kind=='h':
            doc.add_heading(block[1],2);story.append(Paragraph(escape(block[1]),styles['SubheadCompact']))
        elif kind=='image':
            path=WORK/'figures'/block[1];doc.add_picture(str(path),width=Inches(block[2]))
            story.append(PDFImage(str(path),width=block[2]*72,height=block[3]*72));story.append(Spacer(1,7))
        elif kind=='table':
            values=block[1];table=doc.add_table(rows=1,cols=len(values[0]));table.style='Light Shading Accent 1'
            for cell,value in zip(table.rows[0].cells,values[0]):cell.text=value
            for row in values[1:]:
                for cell,value in zip(table.add_row().cells,row):cell.text=value
            for row in table.rows:
                for cell in row.cells:
                    for paragraph in cell.paragraphs:
                        for run in paragraph.runs:run.font.size=Pt(9)
            available=6.95*72;columns=len(values[0]);widths=[available/columns]*columns
            if columns==2:widths=[available*.29,available*.71]
            pdf_rows=[[Paragraph(escape(value),styles['TableCompact']) for value in row] for row in values]
            table_pdf=Table(pdf_rows,colWidths=widths,hAlign='LEFT',repeatRows=1)
            table_pdf.setStyle(TableStyle([('BACKGROUND',(0,0),(-1,0),colors.HexColor('#dbeafe')),('VALIGN',(0,0),(-1,-1),'TOP'),('GRID',(0,0),(-1,-1),.3,colors.HexColor('#94a3b8')),('LEFTPADDING',(0,0),(-1,-1),5),('RIGHTPADDING',(0,0),(-1,-1),5),('TOPPADDING',(0,0),(-1,-1),4),('BOTTOMPADDING',(0,0),(-1,-1),2)]))
            story.append(table_pdf);story.append(Spacer(1,8))
doc.save(WORK/'report/MGSC673_Footwear_Report.docx')
def report_footer(canvas,document):
    canvas.setFont('MGSCBody',8)
    canvas.drawString(55,23,'MGSC 673 | Dora Zhuang')
    canvas.drawRightString(557,23,str(canvas.getPageNumber()))
SimpleDocTemplate(str(WORK/'report/MGSC673_Footwear_Report.pdf'),title='Footwear patterns under occlusion and cross-medium transfer',author='Dora Zhuang',pagesize=(612,792),leftMargin=55,rightMargin=55,topMargin=42,bottomMargin=42).build(story,onFirstPage=report_footer,onLaterPages=report_footer)
print('Report saved in Word, PDF and Markdown:',WORK/'report')
print('The report, figures and visible notebook results are saved alongside their source tables.')


## 9. Outputs
The extracted project contains all 308 prepared RGB224 inputs and the exact source manifests. Default execution verifies and reuses the archived GPU models; the optional re-training check writes to a separate directory and never replaces the formal experiment. Tables, figures and the report are written to `reproduced_analysis/`. Run and packaging instructions are in README.md.


In [ ]:
print('Analysis complete.')
print('Tables, figures and report:',WORK)
print('Report files:',WORK/'report')
print('Formal GPU models, original manifests and locks were not modified.')
